In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib as path
import plotly.express as px


In [24]:

file_path = r"C:\Users\heryn\OneDrive\Documents\Semester 1 2026\FIT 5120\iteration 1 brainboost\project brain health\Measured-physical-activity-and-sleep-all\MPASDC02.xlsx"

df = pd.read_excel(file_path, sheet_name="Table 1.1_Proportions", header=None)

print(df.head(20))

                                                   0   \
0   This tab contains table 1.1, proportion of per...   
1                     Australian Bureau of Statistics   
2   Table 1.1 Average hours of sleep per night(a),...   
3   National Nutrition and Physical Activity Surve...   
4                                                 NaN   
5                                                 NaN   
6                                                 NaN   
7             Average hours of sleep per weeknight(e)   
8                                   Less than 6 hours   
9                              6 to less than 7 hours   
10                             7 to less than 8 hours   
11                             8 to less than 9 hours   
12                            9 to less than 10 hours   
13                                   10 hours or more   
14                                      Total persons   
15        Average hours of sleep per weekend night(f)   
16                             

In [25]:
# use row 5 for age-group column names
columns = ["category"] + df.iloc[5, 1:].tolist()

# keep the table body only
table = df.iloc[7:38, 0:len(columns)].copy()
table.columns = columns

# clean column names
table.columns = [str(col).strip() for col in table.columns]

# clean category text
table["category"] = table["category"].astype(str).str.strip()

# keep only 18–24 column
sleep_18_24 = table[["category", "18–24"]].copy()
sleep_18_24 = sleep_18_24.rename(columns={"18–24": "value"})

# remove blank rows
sleep_18_24 = sleep_18_24.dropna(subset=["category"])

# remove footnotes, notes, and copyright rows
sleep_18_24 = sleep_18_24[
    ~sleep_18_24["category"].str.contains(
        "Footnotes|Commonwealth of Australia|Proportions may not add up|^\\(",
        case=False,
        na=False,
        regex=True
    )
]

# clean value column
sleep_18_24["value"] = (
    sleep_18_24["value"]
    .astype(str)
    .str.replace("#", "", regex=False)
    .str.replace(",", ".", regex=False)
)

# convert numeric where possible
sleep_18_24["value"] = pd.to_numeric(sleep_18_24["value"], errors="coerce")

print(sleep_18_24.head(30))

                                             category  value
7             Average hours of sleep per weeknight(e)    NaN
8                                   Less than 6 hours   10.3
9                              6 to less than 7 hours   18.9
10                             7 to less than 8 hours   33.3
11                             8 to less than 9 hours   27.3
12                            9 to less than 10 hours    9.8
13                                   10 hours or more    0.4
14                                      Total persons  100.0
15        Average hours of sleep per weekend night(f)    NaN
16                                  Less than 6 hours   17.7
17                             6 to less than 7 hours    9.0
18                             7 to less than 8 hours   25.7
19                             8 to less than 9 hours   25.7
20                            9 to less than 10 hours    9.7
21                                   10 hours or more   12.2
22                      

In [30]:
# remove footer / notes rows
sleep_18_24_clean = sleep_18_24[
    ~sleep_18_24["category"].astype(str).str.startswith("#")
]

sleep_18_24_clean = sleep_18_24_clean[
    ~sleep_18_24_clean["category"].astype(str).str.startswith("(")
]

sleep_18_24_clean = sleep_18_24_clean[
    ~sleep_18_24_clean["category"].astype(str).str.contains(
        "Proportions may not add up",
        case=False,
        na=False
    )
].copy()

print(sleep_18_24_clean)

                                       category  value
7       Average hours of sleep per weeknight(e)    NaN
8                             Less than 6 hours   10.3
9                        6 to less than 7 hours   18.9
10                       7 to less than 8 hours   33.3
11                       8 to less than 9 hours   27.3
12                      9 to less than 10 hours    9.8
13                             10 hours or more    0.4
14                                Total persons  100.0
15  Average hours of sleep per weekend night(f)    NaN
16                            Less than 6 hours   17.7
17                       6 to less than 7 hours    9.0
18                       7 to less than 8 hours   25.7
19                       8 to less than 9 hours   25.7
20                      9 to less than 10 hours    9.7
21                             10 hours or more   12.2
22                                Total persons  100.0
23             Average hours of sleep per night    NaN
24        

In [46]:
weeknight_sleep = sleep_18_24_clean.iloc[0:6].copy()
weekend_sleep = sleep_18_24_clean.iloc[6:12].copy()
overall_sleep = sleep_18_24_clean.iloc[12:24].copy()

In [47]:
sleep_combined = pd.concat(
    [weeknight_sleep, weekend_sleep, overall_sleep],
    ignore_index=True
)

print(sleep_combined)

                                       category  value
0       Average hours of sleep per weeknight(e)    NaN
1                             Less than 6 hours   10.3
2                        6 to less than 7 hours   18.9
3                        7 to less than 8 hours   33.3
4                        8 to less than 9 hours   27.3
5                       9 to less than 10 hours    9.8
6                              10 hours or more    0.4
7                                 Total persons  100.0
8   Average hours of sleep per weekend night(f)    NaN
9                             Less than 6 hours   17.7
10                       6 to less than 7 hours    9.0
11                       7 to less than 8 hours   25.7
12                       8 to less than 9 hours   25.7
13                      9 to less than 10 hours    9.7
14                             10 hours or more   12.2
15                                Total persons  100.0
16             Average hours of sleep per night    NaN
17        

In [48]:
weeknight_sleep["sleep_type"] = "Weeknight"
weekend_sleep["sleep_type"] = "Weekend night"
overall_sleep["sleep_type"] = "Overall night"

sleep_combined = pd.concat(
    [weeknight_sleep, weekend_sleep, overall_sleep],
    ignore_index=True
)

print(sleep_combined)

                                       category  value     sleep_type
0       Average hours of sleep per weeknight(e)    NaN      Weeknight
1                             Less than 6 hours   10.3      Weeknight
2                        6 to less than 7 hours   18.9      Weeknight
3                        7 to less than 8 hours   33.3      Weeknight
4                        8 to less than 9 hours   27.3      Weeknight
5                       9 to less than 10 hours    9.8      Weeknight
6                              10 hours or more    0.4  Weekend night
7                                 Total persons  100.0  Weekend night
8   Average hours of sleep per weekend night(f)    NaN  Weekend night
9                             Less than 6 hours   17.7  Weekend night
10                       6 to less than 7 hours    9.0  Weekend night
11                       7 to less than 8 hours   25.7  Weekend night
12                       8 to less than 9 hours   25.7  Overall night
13                  

In [50]:
import pandas as pd
import plotly.express as px

# start from your dataframe
sleep_graph = sleep_combined.copy()

# keep only actual sleep-duration categories
sleep_graph = sleep_graph[
    sleep_graph["category"].isin([
        "Less than 6 hours",
        "6 to less than 7 hours",
        "7 to less than 8 hours",
        "8 to less than 9 hours",
        "9 to less than 10 hours",
        "10 hours or more"
    ])
].copy()

# make sure value is numeric
sleep_graph["value"] = pd.to_numeric(sleep_graph["value"], errors="coerce")

# optional: set category order
category_order = [
    "Less than 6 hours",
    "6 to less than 7 hours",
    "7 to less than 8 hours",
    "8 to less than 9 hours",
    "9 to less than 10 hours",
    "10 hours or more"
]

sleep_graph["category"] = pd.Categorical(
    sleep_graph["category"],
    categories=category_order,
    ordered=True
)

# interactive grouped bar chart
fig = px.bar(
    sleep_graph,
    x="category",
    y="value",
    color="sleep_type",
    barmode="group",
    text="value",
    title="Sleep Duration Distribution for Ages 18–24",
    labels={
        "category": "Sleep duration",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    hovertemplate="Sleep type: %{fullData.name}<br>Duration: %{x}<br>Percentage: %{y:.1f}%<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    xaxis_tickangle=-30
)

fig.show()

In [56]:
import plotly.express as px
import pandas as pd

sleep_graph = sleep_combined.copy()

sleep_graph = sleep_graph[
    sleep_graph["category"].isin([
        "Less than 6 hours",
        "6 to less than 7 hours",
        "7 to less than 8 hours",
        "8 to less than 9 hours",
        "9 to less than 10 hours",
        "10 hours or more"
    ])
].copy()

sleep_graph["value"] = pd.to_numeric(sleep_graph["value"], errors="coerce")

fig = px.bar(
    sleep_graph,
    x="category",
    y="value",
    color="sleep_type",
    facet_row="sleep_type",
    text="value",
    title="Sleep Duration Distribution for Ages 18–24",
    labels={
        "category": "Sleep duration",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    height=850,
    title_x=0.5,
    showlegend=False
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(tickangle=-25)

fig.show()

In [59]:
import re
import pandas as pd
import plotly.express as px

# start from your existing dataframe
sleep_graph = sleep_combined.copy()

# keep only real sleep-duration rows
sleep_graph = sleep_graph[
    sleep_graph["category"].astype(str).str.contains(
        "Less than|to less than|hours or more",
        na=False
    )
].copy()

sleep_graph["value"] = pd.to_numeric(sleep_graph["value"], errors="coerce")

# ---------- derive midpoint hours from the category text ----------
def estimate_midpoint(label: str) -> float:
    text = str(label).strip().lower()

    # case 1: "6 to less than 7 hours"
    m = re.search(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)", text)
    if m:
        low = float(m.group(1))
        high = float(m.group(2))
        return (low + high) / 2

    # case 2: "Less than 6 hours"
    m = re.search(r"less than\s+(\d+(?:\.\d+)?)", text)
    if m:
        high = float(m.group(1))
        return high - 0.5

    # case 3: "10 hours or more"
    m = re.search(r"(\d+(?:\.\d+)?)\s+hours?\s+or\s+more", text)
    if m:
        low = float(m.group(1))
        return low + 0.5

    return None

sleep_graph["midpoint_hours"] = sleep_graph["category"].apply(estimate_midpoint)

# ---------- weighted average hours by sleep type ----------
avg_sleep = (
    sleep_graph.groupby("sleep_type", as_index=False)
    .apply(lambda x: pd.Series({
        "avg_hours": (x["midpoint_hours"] * x["value"]).sum() / x["value"].sum()
    }))
    .reset_index(drop=True)
)

print(avg_sleep)

# ---------- cleaner short labels ----------
def short_label(label: str) -> str:
    text = str(label).strip()
    text = text.replace("Less than ", "<")
    text = text.replace(" to less than ", "–<")
    text = text.replace(" hours or more", "+ h")
    text = text.replace(" hours", " h")
    return text

sleep_graph["category_short"] = sleep_graph["category"].apply(short_label)

# order categories from shortest to longest using midpoint
category_order = (
    sleep_graph[["category_short", "midpoint_hours"]]
    .drop_duplicates()
    .sort_values("midpoint_hours")["category_short"]
    .tolist()
)

# merge averages back for subplot labels
sleep_graph = sleep_graph.merge(avg_sleep, on="sleep_type", how="left")

# build nicer facet titles
title_map = {
    row["sleep_type"]: f'{row["sleep_type"]} · estimated average = {row["avg_hours"]:.2f} h'
    for _, row in avg_sleep.iterrows()
}

sleep_graph["facet_label"] = sleep_graph["sleep_type"].map(title_map)

# ---------- plot ----------
fig = px.bar(
    sleep_graph,
    x="category_short",
    y="value",
    color="sleep_type",
    facet_row="facet_label",
    text="value",
    category_orders={"category_short": category_order},
    title="Sleep Duration Distribution for Ages 18–24",
    labels={
        "category_short": "Sleep duration band",
        "value": "Percentage"
    },
    hover_data={
        "sleep_type": True,
        "value": ":.1f",
        "midpoint_hours": ":.1f",
        "avg_hours": ":.2f"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "Sleep type: %{customdata[0]}<br>"
        "Band: %{x}<br>"
        "Percentage: %{y:.1f}%<br>"
        "Band midpoint: %{customdata[2]:.1f} h<br>"
        "Estimated average for this sleep type: %{customdata[3]:.2f} h"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=900,
    title_x=0.5,
    showlegend=False
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(tickangle=0)
fig.update_yaxes(range=[0, max(sleep_graph["value"]) + 8])

fig.show()

      sleep_type  avg_hours
0  Weekend night   7.590543
1      Weeknight   7.507973


C:\Users\heryn\AppData\Local\Temp\ipykernel_58852\545329567.py:48: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [60]:
import re
import pandas as pd
import plotly.express as px

# start from your actual dataframe
sleep_graph = sleep_combined.copy()

# keep only the real sleep bands
sleep_graph = sleep_graph[
    sleep_graph["category"].astype(str).str.contains(
        "Less than|to less than|hours or more",
        na=False
    )
].copy()

sleep_graph["value"] = pd.to_numeric(sleep_graph["value"], errors="coerce")

# turn category text into numeric midpoint for plotting
def estimate_midpoint(label):
    text = str(label).strip().lower()

    m = re.search(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)", text)
    if m:
        low = float(m.group(1))
        high = float(m.group(2))
        return (low + high) / 2

    m = re.search(r"less than\s+(\d+(?:\.\d+)?)", text)
    if m:
        return float(m.group(1)) - 0.5

    m = re.search(r"(\d+(?:\.\d+)?)\s+hours?\s+or\s+more", text)
    if m:
        return float(m.group(1)) + 0.5

    return None

sleep_graph["midpoint_hours"] = sleep_graph["category"].apply(estimate_midpoint)

# shorter labels for hover
def short_label(label):
    text = str(label).strip()
    text = text.replace("Less than ", "<")
    text = text.replace(" to less than ", "–<")
    text = text.replace(" hours or more", "+ h")
    text = text.replace(" hours", " h")
    return text

sleep_graph["category_short"] = sleep_graph["category"].apply(short_label)

# line chart
fig = px.line(
    sleep_graph,
    x="midpoint_hours",
    y="value",
    color="sleep_type",
    markers=True,
    title="Sleep Duration Distribution for Ages 18–24",
    labels={
        "midpoint_hours": "Estimated sleep hours",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    },
    hover_data={
        "category_short": True,
        "value": ":.1f"
    }
)

fig.update_traces(
    mode="lines+markers",
    hovertemplate=(
        "Sleep type: %{fullData.name}<br>"
        "Band: %{customdata[0]}<br>"
        "Percentage: %{y:.1f}%<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=600,
    title_x=0.5,
    xaxis=dict(
        tickmode="array",
        tickvals=[5.5, 6.5, 7.5, 8.5, 9.5, 10.5],
        ticktext=["<6 h", "6–<7 h", "7–<8 h", "8–<9 h", "9–<10 h", "10+ h"]
    )
)

fig.show()

In [63]:
import re
import pandas as pd
import plotly.express as px

sleep_graph = sleep_combined.copy()

sleep_graph = sleep_graph[
    sleep_graph["category"].astype(str).str.contains(
        "Less than|to less than|hours or more",
        na=False
    )
].copy()

sleep_graph["value"] = pd.to_numeric(sleep_graph["value"], errors="coerce")

def estimate_midpoint(label):
    text = str(label).strip().lower()

    m = re.search(r"(\d+(?:\.\d+)?)\s+to\s+less\s+than\s+(\d+(?:\.\d+)?)", text)
    if m:
        low = float(m.group(1))
        high = float(m.group(2))
        return (low + high) / 2

    m = re.search(r"less than\s+(\d+(?:\.\d+)?)", text)
    if m:
        return float(m.group(1)) - 0.5

    m = re.search(r"(\d+(?:\.\d+)?)\s+hours?\s+or\s+more", text)
    if m:
        return float(m.group(1)) + 0.5

    return None

sleep_graph["midpoint_hours"] = sleep_graph["category"].apply(estimate_midpoint)

def short_label(label):
    text = str(label).strip()
    text = text.replace("Less than ", "<")
    text = text.replace(" to less than ", "–<")
    text = text.replace(" hours or more", "+ h")
    text = text.replace(" hours", " h")
    return text

sleep_graph["category_short"] = sleep_graph["category"].apply(short_label)

avg_sleep = (
    sleep_graph.groupby("sleep_type")
    .apply(lambda x: (x["midpoint_hours"] * x["value"]).sum() / x["value"].sum())
    .reset_index(name="avg_hours")
)

print(avg_sleep)

avg_map = dict(zip(avg_sleep["sleep_type"], avg_sleep["avg_hours"]))

subtitle_parts = []
if "Weeknight" in avg_map:
    subtitle_parts.append(f"Weeknight: {avg_map['Weeknight']:.2f} h")
if "Weekend night" in avg_map:
    subtitle_parts.append(f"Weekend night: {avg_map['Weekend night']:.2f} h")
if "Overall night" in avg_map:
    subtitle_parts.append(f"Overall night: {avg_map['Overall night']:.2f} h")

subtitle = " | ".join(subtitle_parts)

fig = px.line(
    sleep_graph,
    x="midpoint_hours",
    y="value",
    color="sleep_type",
    markers=True,
    title=f"Sleep Duration Distribution for Ages 18–24<br><sup>{subtitle}</sup>",
    labels={
        "midpoint_hours": "Estimated sleep hours",
        "value": "Percentage",
        "sleep_type": "Sleep type"
    },
    hover_data={
        "category_short": True,
        "value": ":.1f"
    }
)

fig.update_traces(
    mode="lines+markers",
    hovertemplate=(
        "Sleep type: %{fullData.name}<br>"
        "Band: %{customdata[0]}<br>"
        "Percentage: %{y:.1f}%<extra></extra>"
    )
)

for _, row in avg_sleep.iterrows():
    fig.add_vline(
        x=row["avg_hours"],
        line_dash="dash",
        line_width=2
    )

fig.update_layout(
    template="plotly_white",
    height=600,
    title_x=0.5,
    xaxis=dict(
        tickmode="array",
        tickvals=[5.5, 6.5, 7.5, 8.5, 9.5, 10.5],
        ticktext=["<6 h", "6–<7 h", "7–<8 h", "8–<9 h", "9–<10 h", "10+ h"]
    )
)

fig.show()

      sleep_type  avg_hours
0  Weekend night   7.590543
1      Weeknight   7.507973


C:\Users\heryn\AppData\Local\Temp\ipykernel_58852\4073083684.py:49: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

